# Divisão espacial k-fold dos patches por fonte (estágio 07)

Atribui cada patch registrado no manifesto de cada fonte do estágio 06 a uma das `splits.fold_count` dobras (k=5) por proximidade geográfica: os centroides das bboxes são agrupados com k-means determinístico e os grupos são rotulados pela posição dos centros. A contiguidade espacial dos grupos reduz o vazamento por autocorrelação entre treino e validação nos estágios 09/10 — dentro de cada conjunto de patches da fonte.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt`, garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo a subpasta do manifesto e das figuras.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
print(f"Plataforma: {platform}")
print(f"Manifestos por fonte: {storage_paths['data_processed']}/manifest_<fonte>.parquet")
print(f"Figuras: {storage_paths['artifacts_figures']}")


## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependência do estágio 06 (manifestos por fonte)

Verifica que o manifesto de patches de cada fonte do estágio 06 está disponível no caminho canônico e informa se ele está vigente frente às entradas atuais (composite e máscara da fonte).

In [ ]:
# Verifica a dependência do estágio 06 (manifesto de patches por fonte).
from src.data.patch_generation import manifest_is_current, manifest_path
from src.utils import check_dependencies

enabled_sources = [
    name for name, cfg in config["ground_truth"]["sources"].items() if cfg["enabled"]
]
for name in enabled_sources:
    manifest_file = manifest_path(storage_paths, name)
    check_dependencies({f"Manifesto {name} (estágio 06)": manifest_file})
    print(f"Manifesto {name} vigente frente às entradas: {manifest_is_current(storage_paths, name)}")


## Divisão espacial k-fold por fonte

Garante (idempotente) a divisão espacial k-fold de cada fonte: os patches do manifesto da fonte são agrupados por proximidade geográfica dos centroides (k-means determinístico) e a coluna `fold` é gravada no manifesto, versionada em `split_<fonte>.meta.json`.

In [ ]:
# Garante a divisão espacial k-fold de cada fonte (reutiliza se já existir).
from src.data.spatial_split import assign_spatial_folds

split_manifests = {name: assign_spatial_folds(storage_paths, name) for name in enabled_sources}


## Verificação das divisões

Confere a divisão persistida de cada fonte: total de patches e tiles, contagem de patches, proporção média de café e localização média (centroide) de cada dobra.

In [ ]:
# Verifica a divisão persistida de cada fonte (contagens e localização por dobra).
from src.data.spatial_split import verify_split

split_stats = {}
for name in enabled_sources:
    split_stats[name] = verify_split(storage_paths, name)
    stats = split_stats[name]
    print(f"{name}: {stats['fold_count']} dobras | {stats['n_patches']} patches | {stats['n_tiles']} tiles")
    for row in stats["per_fold"]:
        print(
            f"  dobra {row['fold']}: {row['n_patches']} patches | café {row['coffee_ratio_mean']:.3f} | "
            f"centroide ({row['centroid_x']:.0f}, {row['centroid_y']:.0f})"
        )


## Figuras da divisão espacial por fonte

Renderiza e persiste, para cada fonte, o mapa de centroides dos patches coloridos por dobra em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam as figuras existentes.

In [ ]:
# Renderiza e persiste a figura da divisão espacial de cada fonte (reutiliza se já existir).
from src.data.spatial_split import save_split_figure

split_figures = {name: save_split_figure(storage_paths, name) for name in enabled_sources}


## Resumo da etapa

Exibe o resumo da divisão espacial k-fold por fonte: fontes processadas, dobras, patches divididos e figuras de registro.

In [ ]:
# Exibe o resumo da etapa de divisão espacial k-fold por fonte.
summary = {
    "Fontes processadas": ", ".join(enabled_sources),
    "Dobras por fonte": next(iter(split_stats.values()))["fold_count"],
    "Patches divididos": {name: stats["n_patches"] for name, stats in split_stats.items()},
    "Figuras da divisão": ", ".join(str(path) for path in split_figures.values()),
}
from src.utils import print_summary

print_summary(summary, "07")
